# Guess the Condition: API workflow example
---
This notebook shows how to use **Guess the Condition** from Python: scan an experiment, play a game, and analyse the guesses, with the same package that is behind the Napari and Fiji tools and the app notebook.

To keep the example self-contained it builds a small synthetic experiment (two conditions that differ in brightness, three biological repeats) and lets a **simulated player** guess. A simulated player can look the answer up; in a real game the condition of an image is never shown. Replace the synthetic folder with your own experiment folder (one folder per condition, one per biological repeat, TIFF images inside) to analyse your own data.

# Step 0: Initialize environment & install dependencies
---

In [ ]:
# @title Initialize environment & install dependencies
import sys

# Metadata
current_version = "0.1.0"
notebook_name = "GuessTheCondition_API_Example"

if "google.colab" in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")
    !pip install -q "guessthecondition @ git+https://github.com/CellMigrationLab/GuessTheCondition"
    from google.colab import drive
    drive.mount("/content/gdrive")
    print("✅ Colab setup done")
else:
    print("✅ Local environment detected. Assuming dependencies are already installed.")

import ipywidgets as widgets
from IPython.display import Image, display, clear_output

import guessthecondition
from guessthecondition import GameError, Session, analyze, describe, load_for_display, read_results, scan_experiment

print("%s %s, guessthecondition %s" % (notebook_name, current_version, guessthecondition.__version__))

# Step 1: Build a synthetic experiment
---

In [ ]:
# @title Build a synthetic experiment
import tempfile
from pathlib import Path

import numpy as np
import tifffile

work = Path(tempfile.mkdtemp(prefix="guessthecondition_demo_"))
rng = np.random.default_rng(3)
for k, condition in enumerate(["Control", "Mutant"]):
    for repeat in ["R1", "R2", "R3"]:
        folder = work / "Experiment" / condition / repeat
        folder.mkdir(parents=True)
        for i in range(8):
            blobs = rng.poisson(40 + 14 * k, (64, 64)).astype(np.uint16)
            tifffile.imwrite(folder / ("FOV%d.tif" % (i + 1)), blobs)
(work / "Results").mkdir()
print(describe(scan_experiment(work / "Experiment")))

# Step 2: Play
---
The game is a `Session`: `serve()` gives the next image (a path and its number), `answer(guess)` records a guess and the decision time, `undo_last()` takes it back. Everything is kept in the results folder, so a session can be reopened with `Session.open(results_folder, user_name)`.

In [ ]:
# @title Play a game with a simulated player
session = Session.create(work / "Experiment", work / "Results", user_name="Simulated", percentage_to_test=50, seed=1)
print("Images to guess:", session.target, "of", session.total)

player = np.random.default_rng(0)
conditions = session.conditions
while session.remaining:
    item = session.serve()                       # {"number", "of", "path"}: nothing that names the condition
    truth = session.table.loc[session.table["image_file"] == str(item["path"].relative_to(session.experiment_dir).as_posix()), "condition"].iloc[0]
    # the simulated player is right 85% of the time (a real player only sees the image)
    guess = truth if player.random() < 0.85 else player.choice(conditions)
    session.answer(guess)
print("Guessed:", session.n_answered)

# Step 3: Analyse
---
`analyze` takes the table of guessed images (`session.table`) and returns an `Analysis`: the readout (marker, chip, text), accuracy against chance, the test across biological repeats, per-repeat and per-condition tables, the confusion matrix and the decision times.

In [ ]:
# @title Analyse
analysis = analyze(session.table)
print(analysis.text())

In [ ]:
# @title Figures, tables and the PDF report
from guessthecondition.report import accuracy_figure, confusion_figure, figure_png, write_report

display(Image(figure_png(confusion_figure(analysis))))
display(Image(figure_png(accuracy_figure(analysis))))
display(analysis.per_repeat)
display(analysis.per_condition)
paths = write_report(analysis, session.answered(), session.folder / "analysis")
print("Report written to", paths["analysis_results.pdf"])

The table in `game_results.csv` is a plain CSV (`image_file`, `condition`, `biological_repeat`, `guess`, `decision_time_s`, ...): any game, from any front end, can be analysed again later, or with another tool.

In [ ]:
# @title Analyse results saved earlier
import pandas as pd

saved = read_results(session.folder / "game_results.csv")      # the table of any game: condition, biological_repeat, guess, decision_time_s
again = analyze(saved)
print(again.readout.marker, again.readout.chip)

# Version log
---
**Version 0.1.0.** First release.